In [1]:
import numpy as np
from collections import Counter

In [12]:
#Decision Tree
class DecisionTree:
    def __init__(self,max_depth=5,min_size=2,n_features=None):
        self.max_depth=max_depth
        self.min_size=min_size
        self.n_features=n_features
        self.tree=None
    def fit(self,x,y):
        self.n_classes=len(set(y))
        self.n_features=self.n_features or x.shape[1]
        self.tree=self._grow(x,y,depth=0)
    def _best_split(self,x,y):
        best_gini= 1e9
        best_idx,best_thr=None,None
        features=np.random.choice(x.shape[1],self.n_features,replace=False)

        for idx in features:
            thresholds=np.unique(x[:,idx])
            for t in thresholds:
                left=y[x[:,idx]<t]
                right=y[x[:,idx]>=t]
                if len(left)==0 or len(right)==0:
                    continue
                gini=self._gini(left,right)
                if gini<best_gini:
                    best_gini=gini
                    best_idx=idx
                    best_thr=t
        return best_idx,best_thr
    def _gini(self,left,right):
        total=len(left)+len(right)
        g=0.0
        for group in (left,right):
            size=len(group)
            if size==0.0:
                continue
            score=0.0
            for c in range(self.n_classes):
                p=np.sum(group==c)/size
                score+=p*p
            g += (1-score)*(size/total)
        return g
    def _grow(self,x,y,depth):
        if len(set(y))==1 or depth >= self.max_depth or len(y)<=self.min_size:
            return Counter(y).most_common(1)[0][0]
        idx,thr=self._best_split(x,y)
        if idx is None:
            return Counter(y).most_common(1)[0][0]
        left_mask=x[:,idx]<thr
        right_mask = x[:, idx] >= thr
        return{
            'idx':idx,
            'thr':thr,
            'left':self._grow(x[left_mask],y[left_mask],depth+1),
            'right':self._grow(x[right_mask],y[right_mask],depth+1)
        }
    def predict_one(self,x,node):
        if not isinstance(node,dict):
                return node
        if x[node['idx']]<node['thr']:
            return self.predict_one(x,node['left'])
        return self.predict_one(x,node['right'])
    def predict(self,x):
        return np.array([self.predict_one(X, self.tree) for X in x])